In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class AtrousConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super(AtrousConv, self).__init__()
        padding = dilation if kernel_size == 3 else 0
        self.atrous_conv = nn.Conv2d(in_channels, out_channels, kernel_size, 
                                     padding=padding, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        x = self.atrous_conv(x)
        x = self.bn(x)
        return self.relu(x)

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ASPP, self).__init__()
        self.conv1x1 = AtrousConv(in_channels, out_channels, kernel_size=1, dilation=1)
        self.conv3x3_r6 = AtrousConv(in_channels, out_channels, kernel_size=3, dilation=6)
        self.conv3x3_r12 = AtrousConv(in_channels, out_channels, kernel_size=3, dilation=12)
        self.conv3x3_r18 = AtrousConv(in_channels, out_channels, kernel_size=3, dilation=18)
        
        # Global pooling layer
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.conv1x1_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        # Final 1x1 conv layer after concatenation
        self.conv1x1_out = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_out = nn.BatchNorm2d(out_channels)
        
    def forward(self, x):
        conv1x1 = self.conv1x1(x)
        conv3x3_r6 = self.conv3x3_r6(x)
        conv3x3_r12 = self.conv3x3_r12(x)
        conv3x3_r18 = self.conv3x3_r18(x)
        
        # Global average pooling
        global_pool = self.global_pool(x)
        global_pool = self.conv1x1_global(global_pool)
        global_pool = self.bn_global(global_pool)
        global_pool = F.interpolate(global_pool, size=x.shape[2:], mode='bilinear', align_corners=False)
        
        # Concatenate all the features
        x = torch.cat([conv1x1, conv3x3_r6, conv3x3_r12, conv3x3_r18, global_pool], dim=1)
        x = self.conv1x1_out(x)
        return self.bn_out(x)

class DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes):
        super(DeepLabV3Plus, self).__init__()
        
        # Backbone (you can replace with a pre-trained ResNet or similar model)
        self.backbone = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-2])  # Remove last FC and avg pool
        
        # ASPP module
        self.aspp = ASPP(in_channels=2048, out_channels=256)
        
        # Upsampling layers
        self.upsample1 = nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1)
        self.upsample2 = nn.ConvTranspose2d(256, num_classes, kernel_size=16, stride=8, padding=4)
        
    def forward(self, x):
        # Backbone
        x = self.backbone(x)
        
        # ASPP
        x = self.aspp(x)
        
        # Upsample to match the input image size
        x = self.upsample1(x)
        x = self.upsample2(x)
        
        return x

# Test the model
model = DeepLabV3Plus(num_classes=21)  # For 21 classes (e.g., PASCAL VOC dataset)
input_tensor = torch.randn(1, 3, 512, 512)  # Batch of 1, 3 channels, 512x512 image
output = model(input_tensor)
print("Output shape:", output.shape)  # Should be [1, num_classes, 512, 512]
